# Building a ChatBot

This chatbot will be able to have a conversation and remember previous interactions. It will only use language model to have a conversation.

- Conversational RAG - Enable a chatbot experience over an external source of data
- Agents - Build a chatbot that can take actions

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
groq_api_key = os.getenv("GROQ_API_KEY")

In [7]:
from langchain_groq import ChatGroq
model = ChatGroq(model="llama-3.1-8b-instant", groq_api_key=groq_api_key)
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000284E9718410>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000284E9718E10>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [8]:
from langchain_core.messages import HumanMessage, AIMessage

model.invoke([HumanMessage(content="Hi, My name is Karan and I'm Learning AIML")])

AIMessage(content="Nice to meet you, Karan. AIML (Artificial Intelligence Markup Language) is a popular platform for building chatbots and virtual assistants. \n\nTo assist you, can you tell me a bit more about what you're trying to learn or accomplish with AIML? Are you looking to create a simple chatbot or a more complex one? Do you have any specific goals or requirements in mind?\n\nAlso, I'd be happy to provide you with some resources, tutorials, or examples to help you get started with AIML. Just let me know how I can assist you.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 117, 'prompt_tokens': 48, 'total_tokens': 165, 'completion_time': 0.195094051, 'completion_tokens_details': None, 'prompt_time': 0.037381644, 'prompt_tokens_details': None, 'queue_time': 0.117949756, 'total_time': 0.232475695}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'm

In [10]:
model.invoke(
    [
        HumanMessage(content="Hi, My name is Karan and I'm Learning AI"),
        AIMessage(content="Hello Karan! That's great to hear. How can I assist you in your AIML learning journey?"),
        HumanMessage(content="Can you tell me my name and what I'm learning?"),
    ]
)

AIMessage(content='Your name is Karan. You are currently learning Artificial Intelligence and Machine Learning (AIML).', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 91, 'total_tokens': 111, 'completion_time': 0.039877298, 'completion_tokens_details': None, 'prompt_time': 0.029175547, 'prompt_tokens_details': None, 'queue_time': 0.091892658, 'total_time': 0.069052845}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_1151d4f23c', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--fd5cd074-27fe-4986-ab1f-d3c77e94b434-0', usage_metadata={'input_tokens': 91, 'output_tokens': 20, 'total_tokens': 111})

### Message History

We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Furture interactions will then load those messages and pass them into the chain as part of the input.

In [14]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(
    model,
    get_session_history
)

In [13]:
config = {"configurable": {"session_id" : "chat1"}}

In [16]:
response = with_message_history.invoke(
    [HumanMessage(content="Hi, My name is Karan and I'm Learning AI")],
    config=config
)

In [17]:
response.content

"Hello Karan. It's great that you're interested in learning AI. What is your current level of knowledge in AI - beginner, intermediate, or advanced? Also, are you using any specific resources or courses to learn AI, such as books, videos, or online platforms?"

In [21]:
with_message_history.invoke(
    [HumanMessage(content="What is my name?")],
    config=config
)

AIMessage(content='Your name is Karan.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 248, 'total_tokens': 255, 'completion_time': 0.004428343, 'completion_tokens_details': None, 'prompt_time': 0.03217404, 'prompt_tokens_details': None, 'queue_time': 0.089756844, 'total_time': 0.036602383}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--652f07da-41e4-43cf-ab25-1316a28ad2b4-0', usage_metadata={'input_tokens': 248, 'output_tokens': 7, 'total_tokens': 255})

In [22]:
config1 = {"configurable": {"session_id" : "chat2"}}
response2 = with_message_history.invoke(
    [HumanMessage(content="What is my name?")],
    config=config1
)
response2.content

"I don't have information about your name. I'm a large language model, I don't have personal knowledge of individual users, and our conversation just started. If you'd like to share your name with me, I'd be happy to chat with you."

# Prompt Templates

Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. 
We will add in system message with some custom instructions (but still taking messages as input). Next,
we'll add in more input besides just the messages

In [26]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant. Answer all the questions to the best of your ability."),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model


In [27]:
chain.invoke({"messages": [HumanMessage(content="Hi, My name is Karan and I'm Learning AI")]})

AIMessage(content="Hello Karan, nice to meet you. I'm thrilled to hear that you're learning AI. It's an exciting and rapidly evolving field, and I'm more than happy to help you with any questions or topics you'd like to explore.\n\nWhat specifically would you like to learn or discuss about AI? Are you interested in machine learning, deep learning, natural language processing, or something else? Let me know, and I'll do my best to assist you.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 94, 'prompt_tokens': 64, 'total_tokens': 158, 'completion_time': 0.160185449, 'completion_tokens_details': None, 'prompt_time': 0.003486835, 'prompt_tokens_details': None, 'queue_time': 0.087845684, 'total_time': 0.163672284}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_1151d4f23c', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--baded89f-9a1d-402c-b6ef-db85ba0af6fe-0', usage_metadata={'i

In [28]:
with_message_history = RunnableWithMessageHistory(chain, get_session_history)

In [29]:
config = {"configurable": {"session_id" : "chat3"}}
response3 = with_message_history.invoke(
    [HumanMessage(content="Hi, My name is Karan and I'm Learning AI")],
    config=config
)

response3.content

"Hello Karan, nice to meet you. Learning AI can be a fascinating and challenging field. I'm happy to assist and guide you through your journey. What specific topics or areas within AI are you interested in learning about?"

In [30]:
## Adding more complexity


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant. Answer all the questions to the best of your ability in {language}"),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model


In [42]:
response4 = chain.invoke({"messages": [HumanMessage(content="Hi, My name is Karan and I'm Learning AI")], "language": "Hinglish"})

In [43]:
response4.content

"Naukari Karan Ji, aapki sambhāl hai! (Congratulations Karan, all the best!) Learning AI, bahut accha hai! (That's great!) Aap kya jante hain, aap AI mein kya sikhna chahte hain? (What do you know, and what do you want to learn in AI?) Main aapko madad karta hoon. (I'll help you.)"

Lets now wrap this more complicated chain in a Message History Class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [44]:
with_message_history = RunnableWithMessageHistory(chain, get_session_history, input_messages_key="messages")

In [45]:
config = {"configurable": {"session_id" : "chat4"}}
response5 = with_message_history.invoke(
    {"messages": [HumanMessage(content="Hi, I am Karan and I'm Learning AI")], "language": "Hinglish"},
    config=config)

response5.content

'Nayi duniya hai, Karan bhai! AI (Artificial Intelligence) padhna ek bahut hi achha vichar hai. Main aapko sabse pehle samarthan karta hoon. Aapne AI ki kya zaroorat hai? Kya aapke paas ek vishesh topic hai jo aapko seekhna hai?'

In [46]:
response6 = with_message_history.invoke(
    {"messages": [HumanMessage(content="Whats my name? And what do i do?")], "language": "Hinglish"},
    config=config)

response6.content

'Karan bhai, aapka naam hai Karan aur aap AI (Artificial Intelligence) seekh rahe hain!'

# Managing the Conversation History

When building chatbots, we need to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.

```trim_messages``` is a helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages.

In [58]:
from langchain_core.messages import SystemMessage, trim_messages

trimmer = trim_messages(
    max_tokens=55,
    strategy="last",
    token_counter=model,
    include_system = True,
    allow_partial = False,
    start_on="human")
messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="Hello, my name is Karan and I'm learning AI."),
    AIMessage(content="Hi!"),
    HumanMessage(content="I like vanilla ice-cream."),
    AIMessage(content="Me too!"),
    HumanMessage(content="Whats is 2+5?"),
    AIMessage(content="7."),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]

trimmer.invoke(messages)


[SystemMessage(content='You are a helpful assistant.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice-cream.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Me too!', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Whats is 2+5?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='7.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={})]

In [60]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

chain = (
    RunnablePassthrough.assign(messages=itemgetter("messages") | trimmer) | prompt | model 
)

response = chain.invoke({
    "messages": messages + [HumanMessage(content="What problem did i ask?")],
    "language": "English"}
)

response.content

'You asked me to add 2 and 5, and I gave you the answer.'